# Monthly Climate Indicators from NLDAS-3 Data (Multiprocessing Version)

This notebook processes daily NLDAS-3 data to calculate monthly Growing Degree Days (GDD), Heat Stress Days (HSD), and Frost Days using multiprocessing for parallel processing of multiple months.

Each monthly file contains all three indicators as variables:
- **gdd**: Growing Degree Days (base 10°C)
- **hsd**: Heat Stress Days (max temp > 30°C) 
- **frost_days**: Frost Days (min temp < 0°C)

Output files are named: `agroclim_indicator-YYYYMM.nc` (e.g., `agroclim_indicator-200001.nc` for January 2000)

In [ ]:
import xarray as xr
import os
import glob
import numpy as np
from datetime import datetime
import pandas as pd
from multiprocessing import Pool, cpu_count
from functools import partial
import warnings
warnings.filterwarnings('ignore')

# Configuration
base_data_dir = "/discover/nobackup/projects/eis_nldas3/DATA/forcing/daily"
output_dir = "/discover/nobackup/projects/eis_nldas3/DATA/forcing/monthly_indicators"

# Define thresholds for calculations
GDD_BASE_TEMP_C = 10    # Growing Degree Days base temperature in Celsius
HSD_THRESHOLD_C = 30    # Heat Stress Days threshold in Celsius
FROST_THRESHOLD_C = 0   # Frost Days threshold in Celsius

# Number of processes to use (adjust based on your system)
NUM_PROCESSES = min(16, cpu_count() - 1)  # Leave one CPU free

print(f"Base data directory: {base_data_dir}")
print(f"Output directory: {output_dir}")
print(f"Number of processes to use: {NUM_PROCESSES}")

In [ ]:
# Discover all month directories
def get_month_directories(base_dir):
    """Get all month directories (YYYYMM format) from the base directory."""
    month_dirs = []
    
    if os.path.exists(base_dir):
        for item in os.listdir(base_dir):
            item_path = os.path.join(base_dir, item)
            # Check if it's a directory and matches YYYYMM format
            if os.path.isdir(item_path) and len(item) == 6 and item.isdigit():
                month_dirs.append(item)
    
    # Sort chronologically
    month_dirs.sort()
    return month_dirs

# Get all month directories
month_directories = get_month_directories(base_data_dir)
print(f"Found {len(month_directories)} month directories")
print(f"First 10 months: {month_directories[:10]}")
print(f"Last 10 months: {month_directories[-10:]}")
print(f"Date range: {month_directories[0]} to {month_directories[-1]}")

In [ ]:
# Create output directory
os.makedirs(output_dir, exist_ok=True)
print(f"Created output directory: {output_dir}")

In [ ]:
# Test with a single month first
if month_directories:
    test_month = month_directories[0]
    print(f"Testing with month: {test_month}")
    
    result = process_single_month(test_month, base_data_dir, output_dir)
    month, success, message = result
    
    if success:
        print(f"✓ Test successful for {month}: {message}")
        
        # Check the output file
        test_file = os.path.join(output_dir, f"agroclim-indicator-{test_month}.nc")
        if os.path.exists(test_file):
            test_ds = xr.open_dataset(test_file)
            print(f"  Output file created: {test_file}")
            print(f"  Variables: {list(test_ds.data_vars)}")
            print(f"  Shape: {dict(test_ds.dims)}")
            test_ds.close()
    else:
        print(f"✗ Test failed for {month}: {message}")

In [ ]:
def process_single_month(month_dir, base_data_dir, output_dir, skip_existing=True):
    """
    Process a single month of NLDAS data to calculate climate indicators.
    
    Args:
        month_dir: Month directory name (YYYYMM format)
        base_data_dir: Base directory containing month directories
        output_dir: Directory to save output files
        skip_existing: If True, skip processing if output file already exists
    
    Returns:
        Tuple of (month_dir, success_bool, message)
    """
    try:
        # Check if output file already exists
        output_filename = f"agroclim-indicator-{month_dir}.nc"
        output_path = os.path.join(output_dir, output_filename)
        
        if skip_existing and os.path.exists(output_path):
            return (month_dir, True, f"Skipped - file already exists: {output_filename}")
        
        month_path = os.path.join(base_data_dir, month_dir)
        
        # Find all daily NetCDF files in the month directory
        files = sorted(glob.glob(os.path.join(month_path, "NLDAS_FOR0010_D.A*.nc")))
        
        if not files:
            return (month_dir, False, f"No NLDAS files found in {month_path}")
        
        # Open all files for this month
        ds = xr.open_mfdataset(files, combine='by_coords', chunks={'time': 'auto'})
        
        # Convert temperatures to Celsius
        ds['tasmin_C'] = ds['Tair_min'] - 273.15
        ds['tasmax_C'] = ds['Tair_max'] - 273.15
        ds['tas_C'] = (ds['tasmin_C'] + ds['tasmax_C']) / 2
        
        # Calculate monthly indicators
        # 1. Growing Degree Days (GDD)
        gdd_daily_contribution = np.maximum(0, ds['tas_C'] - GDD_BASE_TEMP_C)
        gdd_monthly = gdd_daily_contribution.sum(dim='time', skipna=True)
        gdd_monthly.name = 'gdd'
        gdd_monthly.attrs['units'] = 'degC.days'
        gdd_monthly.attrs['long_name'] = f'Monthly Growing Degree Days (base {GDD_BASE_TEMP_C}°C)'
        gdd_monthly.attrs['description'] = f'Sum of daily mean temperature minus {GDD_BASE_TEMP_C}°C for temperatures above base'
        
        # 2. Heat Stress Days (HSD)
        hsd_daily_flag = (ds['tasmax_C'] > HSD_THRESHOLD_C).astype(int)
        hsd_monthly = hsd_daily_flag.sum(dim='time', skipna=True)
        hsd_monthly.name = 'hsd'
        hsd_monthly.attrs['units'] = 'days'
        hsd_monthly.attrs['long_name'] = f'Monthly Heat Stress Days (max temp > {HSD_THRESHOLD_C}°C)'
        hsd_monthly.attrs['description'] = f'Number of days with maximum temperature above {HSD_THRESHOLD_C}°C'
        
        # 3. Frost Days
        frost_daily_flag = (ds['tasmin_C'] < FROST_THRESHOLD_C).astype(int)
        frost_monthly = frost_daily_flag.sum(dim='time', skipna=True)
        frost_monthly.name = 'frost_days'
        frost_monthly.attrs['units'] = 'days'
        frost_monthly.attrs['long_name'] = f'Monthly Frost Days (min temp < {FROST_THRESHOLD_C}°C)'
        frost_monthly.attrs['description'] = f'Number of days with minimum temperature below {FROST_THRESHOLD_C}°C'
        
        # Create combined dataset for this month
        monthly_ds = xr.Dataset({
            'gdd': gdd_monthly,
            'hsd': hsd_monthly,
            'frost_days': frost_monthly
        })
        
        # Add time coordinate with the first day of the month
        year = int(month_dir[:4])
        month = int(month_dir[4:6])
        time_val = pd.Timestamp(year=year, month=month, day=1)
        monthly_ds = monthly_ds.expand_dims({'time': [time_val]})
        
        # Add global attributes
        monthly_ds.attrs['title'] = f'Monthly Climate Indicators for {time_val.strftime("%B %Y")}'
        monthly_ds.attrs['description'] = 'Monthly Growing Degree Days, Heat Stress Days, and Frost Days calculated from NLDAS-3 daily data'
        monthly_ds.attrs['source'] = 'NLDAS-3 daily meteorological data'
        monthly_ds.attrs['created'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        monthly_ds.attrs['gdd_base_temperature'] = f'{GDD_BASE_TEMP_C}°C'
        monthly_ds.attrs['hsd_threshold'] = f'{HSD_THRESHOLD_C}°C'
        monthly_ds.attrs['frost_threshold'] = f'{FROST_THRESHOLD_C}°C'
        monthly_ds.attrs['conventions'] = 'CF-1.6'
        
        # Save to NetCDF file
        monthly_ds.to_netcdf(output_path, 
                           encoding={
                               'gdd': {'zlib': True, 'complevel': 4},
                               'hsd': {'zlib': True, 'complevel': 4},
                               'frost_days': {'zlib': True, 'complevel': 4}
                           })
        
        # Close dataset
        ds.close()
        monthly_ds.close()
        
        return (month_dir, True, f"Successfully processed {len(files)} files")
        
    except Exception as e:
        return (month_dir, False, f"Error: {str(e)}")

In [ ]:
# Process all months using multiprocessing
def process_all_months(month_dirs, base_data_dir, output_dir, num_processes, skip_existing=True):
    """
    Process all months in parallel using multiprocessing.
    
    Args:
        month_dirs: List of month directory names
        base_data_dir: Base directory containing month directories
        output_dir: Directory to save output files
        num_processes: Number of parallel processes to use
        skip_existing: If True, skip processing files that already exist
    """
    print(f"\nProcessing {len(month_dirs)} months using {num_processes} parallel processes...")
    if skip_existing:
        print("Skip existing files: ENABLED")
    else:
        print("Skip existing files: DISABLED - will reprocess all files")
    print("="*60)
    
    # Create partial function with fixed arguments
    process_func = partial(process_single_month, 
                          base_data_dir=base_data_dir, 
                          output_dir=output_dir,
                          skip_existing=skip_existing)
    
    # Process months in parallel
    successful = 0
    failed = 0
    skipped = 0
    failed_months = []
    
    with Pool(processes=num_processes) as pool:
        # Map the process function to all months
        results = pool.map(process_func, month_dirs)
    
    # Process results
    for month, success, message in results:
        if success:
            if "Skipped" in message:
                skipped += 1
                print(f"⏭️  {month}: {message}")
            else:
                successful += 1
                print(f"✓ {month}: {message}")
        else:
            failed += 1
            failed_months.append(month)
            print(f"✗ {month}: {message}")
    
    # Summary
    print("="*60)
    print(f"\nProcessing complete!")
    print(f"  Processed: {successful} months")
    print(f"  Skipped (already exist): {skipped} months")
    print(f"  Failed: {failed} months")
    
    if failed_months:
        print(f"\nFailed months: {failed_months}")
    
    return successful, skipped, failed, failed_months

In [ ]:
# Process all months
# Set skip_existing=False if you want to reprocess all files
SKIP_EXISTING = True  # Change to False to reprocess all files

if month_directories:
    successful, skipped, failed, failed_months = process_all_months(
        month_directories, 
        base_data_dir, 
        output_dir, 
        NUM_PROCESSES,
        skip_existing=SKIP_EXISTING
    )
else:
    print("No month directories found to process!")

In [ ]:
# Process all months
if month_directories:
    successful, failed, failed_months = process_all_months(
        month_directories, 
        base_data_dir, 
        output_dir, 
        NUM_PROCESSES
    )
else:
    print("No month directories found to process!")

In [ ]:
# Optional: Combine all monthly files into a single time series
def combine_monthly_files(output_dir):
    """
    Combine all monthly indicator files into a single time series dataset.
    """
    monthly_files = sorted(glob.glob(os.path.join(output_dir, "agroclim-indicator-*.nc")))
    
    if not monthly_files:
        print("No monthly files found to combine")
        return None
    
    print(f"Combining {len(monthly_files)} monthly files...")
    
    # Open all files and combine along time dimension
    combined_ds = xr.open_mfdataset(monthly_files, combine='by_coords', chunks={'time': 12})
    
    # Sort by time
    combined_ds = combined_ds.sortby('time')
    
    # Save combined dataset
    combined_file = os.path.join(output_dir, "agroclim-indicators-combined.nc")
    
    combined_ds.to_netcdf(combined_file,
                         encoding={
                             'gdd': {'zlib': True, 'complevel': 4},
                             'hsd': {'zlib': True, 'complevel': 4},
                             'frost_days': {'zlib': True, 'complevel': 4}
                         })
    
    print(f"Combined dataset saved to: {combined_file}")
    print(f"Time range: {combined_ds.time.min().values} to {combined_ds.time.max().values}")
    print(f"Number of months: {len(combined_ds.time)}")
    
    combined_ds.close()
    return combined_file

# Uncomment to create combined file
# combined_file = combine_monthly_files(output_dir)

In [ ]:
print("\n" + "="*60)
print("Processing Complete!")
print("="*60)
print(f"Output directory: {output_dir}")
print(f"Total months processed: {len(month_directories)}")
print(f"Output files created: {len(output_files) if 'output_files' in locals() else 0}")
print(f"\nEach file contains:")
print(f"  - gdd: Growing Degree Days (base {GDD_BASE_TEMP_C}°C)")
print(f"  - hsd: Heat Stress Days (max temp > {HSD_THRESHOLD_C}°C)")
print(f"  - frost_days: Frost Days (min temp < {FROST_THRESHOLD_C}°C)")